# UrbanEye - Day 1: Streetlight-Focused Retrain (fast, finishes same day)**GOAL:** Train a model that reliably identifies **streetlight** when you upload/photo a streetlight,using a strong **YOLOv8s** model — WITHOUT spending 5 hours, so it finishes inside today's freeT4 GPU quota and you actually get the files.**The fix for your past failures (CPU + losing files):**1. Cell 0 hard-stops if the GPU is off — no more silent CPU training.2. Only ~1500 images & 30 epochs → finishes in ~30-45 min on T4.3. Traffic: the new files are saved BOTH to Colab download AND Google Drive,   so a disconnect can't erase them.## Before you run1. Runtime > Change runtime type > **T4 GPU**.2. Google Drive mounted? Click the folder icon > Mount drive (needed to auto-save the result).3. Get a FREE Roboflow key: https://app.roboflow.com/settings/api4. Paste the key in the CONFIG cell below. Then **Runtime > Run all**.## What you get at the end- `civic_yolov8.pt`  and  `civic_yolov8.onnx`  (downloaded + saved to Drive)Replace the files in `UrbanEye/backend/ai/models/` with them.

In [ ]:
# ============ CELL 0 - GPU GUARD (run FIRST, never skip) ============# This stops you from silently training on CPU (exactly what wasted 2 days).import subprocessprint(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)import torchassert torch.cuda.is_available(), "NO GPU! Do NOT train on CPU. Fix T4 and re-connect."print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# ============ INSTALL ============!pip install -q -U ultralytics roboflowimport ultralyticsprint("ultralytics", ultralytics.__version__)

In [ ]:
# ================= CONFIG - paste YOUR free Roboflow key =================RF_API_KEY = "PASTE_YOUR_FREE_ROBOFLOW_API_KEY_HERE"# Datasets (street-lamps is the STREETLIGHT source, others keep the 5 classes alive)DATASETS = [    {"tag": "sl", "workspace": "street-lamps", "project": "street-lamps-dpeqc", "version": 1},    {"tag": "gb", "workspace": "garbage-detection-j813v", "project": "garbage-detection-8hmsd", "version": 1},    {"tag": "ph", "workspace": "new-workspace-kj87b", "project": "road-damage-detection-iicdh", "version": 1},    {"tag": "wl", "workspace": "new-workspace-0bgj4", "project": "pipe-leak-yp6il", "version": 1},    {"tag": "dr", "workspace": "sakib-t1srr", "project": "objection-detection-1yrwu", "version": 1},    {"tag": "sw", "workspace": "street-cqv2u", "project": "sidewalk-32xvi", "version": 1},]# Order MUST match the live backend class names.UNIFIED_CLASSES = ["garbage", "pothole", "water_leak", "streetlight", "drainage", "sidewalk_damage"]# ---- DAY-1 STREETLIGHT FOCUS: strong model, small+fast so it finishes ----MODEL = "yolov8s.pt"     # small (strong) modelEPOCHS = 30              # fast on T4 (~30-45 min) - finishes inside quotaIMGSZ = 640FOCUS_CLASS = "streetlight"FOCUS_CAP = 600          # keep up to 600 streetlight imagesOTHER_CAP = 150          # keep only ~150 of each other class (prevent forgetting, stay small)NIGHT_AUG = True         # flip/rotate/scale streetlights so night shots generalizeassert not RF_API_KEY.startswith("PASTE"), "Paste your free Roboflow key"

In [ ]:
# ============ DOWNLOAD DATASETS FROM ROBOFLOW ============from roboflow import Roboflowrf = Roboflow(api_key=RF_API_KEY)downloaded = []for spec in DATASETS:    ds = (rf.workspace(spec["workspace"])            .project(spec["project"])            .version(spec["version"])            .download("yolov8"))    downloaded.append({"spec": spec, "location": ds.location})    print(f"[{spec['tag']}] ready at {ds.location}")

In [ ]:
# ============ MERGE INTO ONE UNIFIED DATASET ============import glob, os, shutil, yamlMERGED = "/content/merged"IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")KEYWORD_TO_UNIFIED = {    "garb": "garbage", "trash": "garbage", "waste": "garbage", "litter": "garbage",    "poth": "pothole", "crack": "pothole", "road damage": "pothole",    "leak": "water_leak", "pipe leak": "water_leak", "water": "water_leak",    "street-lamp": "streetlight", "street light": "streetlight",    "lightpost": "streetlight", "asimetrica": "streetlight",    "drain": "drainage", "sewer": "drainage", "manhole": "drainage", "hole": "drainage",    "sidewalk": "sidewalk_damage", "edge break": "sidewalk_damage",    "joint": "sidewalk_damage", "metal grate": "sidewalk_damage", "patch": "sidewalk_damage",}FOCUS = UNIFIED_CLASSES.index(FOCUS_CLASS)def unify_index(raw_name):    n = str(raw_name).lower()    for kw, unified in KEYWORD_TO_UNIFIED.items():        if kw in n:            return UNIFIED_CLASSES.index(unified)    return Nonedef find_image(images_dir, stem):    for ext in IMG_EXTS:        p = os.path.join(images_dir, stem + ext)        if os.path.exists(p):            return p    return Nonefor split in ("train", "valid"):    os.makedirs(f"{MERGED}/images/{split}", exist_ok=True)    os.makedirs(f"{MERGED}/labels/{split}", exist_ok=True)stats = {c: 0 for c in UNIFIED_CLASSES}for entry in downloaded:    tag = entry["spec"]["tag"]    root = entry["location"]    with open(os.path.join(root, "data.yaml")) as f:        names = yaml.safe_load(f)["names"]    if isinstance(names, dict):        names = [names[k] for k in sorted(names)]    id_map = [unify_index(n) for n in names]    print(f"[{tag}] source -> unified: {list(zip(names, id_map))}")    for split in ("train", "valid", "test"):        out_split = "train" if split == "train" else "valid"        img_src = os.path.join(root, split, "images")        lbl_src = os.path.join(root, split, "labels")        if not os.path.isdir(lbl_src):            continue        for lbl_path in glob.glob(os.path.join(lbl_src, "*.txt")):            stem = os.path.splitext(os.path.basename(lbl_path))[0]            img_path = find_image(img_src, stem)            if img_path is None:                continue            new_lines, used = [], set()            for line in open(lbl_path):                parts = line.split()                if len(parts) != 5:                    continue                mapped = id_map[int(parts[0])]                if mapped is None:                    continue                new_lines.append(" ".join([str(mapped)] + parts[1:]))                used.add(mapped)            if not new_lines:                continue            new_stem = f"{tag}_{stem}"            shutil.copy(img_path, f"{MERGED}/images/{out_split}/{new_stem}{os.path.splitext(img_path)[1]}")            with open(f"{MERGED}/labels/{out_split}/{new_stem}.txt", "w") as f:                f.write("\n".join(new_lines) + "\n")            for c in used:                stats[UNIFIED_CLASSES[c]] += 1print("Boxes per class:")for k, v in stats.items():    print(f"  {k:16s} {v}")

In [ ]:
# ============ STREETLIGHT-FOCUSED BALANCE ============# Keep lots of streetlight, keep a little of everything else so the model# doesn't FORGET potholes/garbage/etc (forgetting would break the app).import glob, os, shutil, randomdef label_path_for(img, split):    stem = os.path.splitext(os.path.basename(img))[0]    return f"{MERGED}/labels/{split}/{stem}.txt"random.seed(7)TRAIN_IMG = f"{MERGED}/images/train"groups = {c: [] for c in UNIFIED_CLASSES}multi = []for img in sorted(glob.glob(TRAIN_IMG + "/*")):    lbl = label_path_for(img, "train")    if not os.path.exists(lbl):        continue    ids = {int(l.split()[0]) for l in open(lbl) if len(l.split()) == 5}    present = {UNIFIED_CLASSES[i] for i in ids}    if len(present) == 1:        groups[list(present)[0]].append(img)    else:        multi.append(img)# Focus class: KEEP THE MOST (cap at FOCUS_CAP)for img in groups[FOCUS_CLASS][FOCUS_CAP:]:    os.remove(img)    lbl = label_path_for(img, "train")    if os.path.exists(lbl):        os.remove(lbl)groups[FOCUS_CLASS] = groups[FOCUS_CLASS][:FOCUS_CAP]# Other classes: keep only a small sample (prevent forgetting, stay fast)for c in UNIFIED_CLASSES:    if c == FOCUS_CLASS:        continue    random.shuffle(groups[c])    for img in groups[c][OTHER_CAP:]:        os.remove(img)        lbl = label_path_for(img, "train")        if os.path.exists(lbl):            os.remove(lbl)    groups[c] = groups[c][:OTHER_CAP]for c in UNIFIED_CLASSES:    print(f"[kept] {c:16s} {len(groups[c])}")import collectionsfinal = collections.Counter()for img in glob.glob(TRAIN_IMG + "/*"):    ids = {int(l.split()[0]) for l in open(label_path_for(img, "train")) if len(l.split()) == 5}    for i in ids:        final[UNIFIED_CLASSES[i]] += 1print("Final train instances:", dict(final))print("Total train images:", len(glob.glob(TRAIN_IMG + "/*")))

In [ ]:
# ============ WRITE data.yaml + SANITY ============data_yaml = {    "path": MERGED,    "train": "images/train",    "val": "images/valid",    "names": {i: c for i, c in enumerate(UNIFIED_CLASSES)},}with open(f"{MERGED}/data.yaml", "w") as f:    yaml.dump(data_yaml, f)train_n = len(glob.glob(f"{MERGED}/images/train/*"))val_n = len(glob.glob(f"{MERGED}/images/valid/*"))sl_val = 0for lbl in glob.glob(f"{MERGED}/labels/valid/*.txt"):    for line in open(lbl):        parts = line.split()        if len(parts) == 5 and int(parts[0]) == UNIFIED_CLASSES.index(FOCUS_CLASS):            sl_val += 1            breakprint(f"train={train_n}  valid={val_n}  streetlight-boxes-in-valid~{sl_val}")assert train_n > 150 and val_n > 50, "Too few images"

In [ ]:
# ============ TRAIN yolov8s (streetlight-focused) ============from ultralytics import YOLOmodel = YOLO(MODEL)          # yolov8s.pt - auto-downloads COCO-pretrained# Night-friendliness: modest augmentation so dark/angled streetlights generalizemodel.train(    data=f"{MERGED}/data.yaml",    epochs=EPOCHS,            # 30 (fast - finishes today)    imgsz=IMGSZ,    batch=16,    patience=10,    cos_lr=True,    seed=7,    augment=True,    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,    degrees=15.0, translate=0.1, scale=0.5, fliplr=0.5,    name="civic_sl",          # distinct run name    project="/content/runs",)BEST = "/content/runs/civic_sl/weights/best.pt"print("Best weights:", BEST)

In [ ]:
# ============ VALIDATE (per class) ============from ultralytics import YOLObest = YOLO(BEST)m = best.val(data=f"{MERGED}/data.yaml", conf=0.30)print(f"mAP@50    : {m.box.map50:.3f}")print(f"mAP@50-95 : {m.box.map:.3f}")n = len(m.box.ap50) if hasattr(m.box, "ap50") else 0for i in range(n):    print(f"  {UNIFIED_CLASSES[i]:16s} mAP50={m.box.ap50[i]:.3f}")

In [ ]:
# ============ KEY TEST: upload a streetlight photo -> must say 'streetlight' ============from google.colab import filesfrom IPython.display import displayfrom PIL import Imageimport ioprint("Upload 2-3 streetlight photos (day + night if you have them):")uploaded = files.upload()for name in uploaded:    r = best.predict(source=name, conf=0.30, verbose=False)[0]    dets = sorted([(best.names[int(b.cls[0])], round(float(b.conf[0]), 2)) for b in r.boxes],                  key=lambda t: -t[1])    top = dets[0] if dets else None    if top and top[0] == "streetlight":        verdict = "OK - correctly identified as STREETLIGHT"    else:        verdict = f"BAD - got {top if top else 'nothing'}"    print(f"{name} -> {dets if dets else 'NO DETECTION'}  |  {verdict}")    if len(r.boxes):        plotted = r.plot()          # BGR numpy        display(Image.fromarray(plotted[:, :, ::-1]))

In [ ]:
# ============ EXPORT + DOWNLOAD (and save to Drive) ============import shutil, osfrom google.colab import filesshutil.copy(BEST, "/content/civic_yolov8.pt")best.export(format="onnx", opset=12, simplify=True)onnx_src = glob.glob("/content/runs/civic_sl/weights/best.onnx")shutil.copy(onnx_src[0], "/content/civic_yolov8.onnx")# Save to Drive too so a disconnect/lost Colab tab can't wipe the result.DRIVE_DIR = "/content/drive/MyDrive/UrbanEye_models"try:    from google.colab import drive    drive.mount("/content/drive")    os.makedirs(DRIVE_DIR, exist_ok=True)    shutil.copy("/content/civic_yolov8.pt", f"{DRIVE_DIR}/civic_yolov8.pt")    shutil.copy("/content/civic_yolov8.onnx", f"{DRIVE_DIR}/civic_yolov8.onnx")    print("Saved to Google Drive:", DRIVE_DIR)except Exception as e:    print("Drive save skipped (still downloading directly):", e)print("\n=== DOWNLOAD THE TWO FILES NOW ===")files.download("/content/civic_yolov8.pt")files.download("/content/civic_yolov8.onnx")print("Downloaded civic_yolov8.pt + civic_yolov8.onnx - replace them in backend/ai/models/")

## After training - deploy YES/NO1. Copy the two downloaded files over:   - `civic_yolov8.onnx` -> `UrbanEye/backend/ai/models/civic_yolov8.onnx`   - `civic_yolov8.pt`   -> `UrbanEye/backend/ai/models/civic_yolov8.pt`2. Tell the assistant "model updated" - it will re-verify the live backend   on your real streetlight photo and confirm it prints "streetlight".Heads-up: this Day-1 run is streetlight-focused but keeps the other 5 classesso they are NOT forgotten. Streetlight is the strong class; others stayworks-but-sharper-later. On Day 2 you can repeat for pothole.